<!-- AI-og-helse: colab-kontrakt v1 -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke04-generativ-ai/oppgaver/prompt_workshop.ipynb)

## Colab-kjøring

- **Anbefalt runtime:** CPU
- **Forventet kjøretid:** 20-40 min
- **Datamønster:** `api-secrets`

**Krav før kjøring:**
- API: `OPENAI_API_KEY`, `GOOGLE_API_KEY` eller `GEMINI_API_KEY` for live-kjøring; kan leses uten i demo-modus
- Data: ingen eksterne datasett
- GPU: ikke nødvendig

**Felles konvensjon:** Kjør setup-cellen rett under først. I Colab hentes hemmeligheter fra **Secrets** med `userdata.get(...)`; lokalt brukes miljøvariabler eller `.env`.

Kjør cellene ovenfra og ned. Setup-cellen installerer bare ekstra pakker når notebooken åpnes i Colab.


In [12]:
# AI-og-helse: Colab bootstrap v1
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_PACKAGES = [
    ('openai', 'openai'),
    ('google-genai', 'google.genai'),
    ('python-dotenv', 'dotenv'),
]
COLAB_DATA_MODE = 'api-secrets'


def _has_import(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def ensure_packages(packages=NOTEBOOK_PACKAGES):
    """Install only notebook-specific packages when running in Colab."""
    if not IN_COLAB:
        return

    missing = [package for package, import_name in packages if not _has_import(import_name)]
    if missing:
        print("Installerer Colab-pakker:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Alle notebook-spesifikke Colab-pakker er tilgjengelige.")


def get_secret(name: str, *aliases: str):
    """Read secrets from environment/.env locally or Colab Secrets in Colab."""
    for key in (name, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[name] = value
            return value

    if IN_COLAB:
        try:
            from google.colab import userdata
        except Exception:
            userdata = None

        if userdata is not None:
            for key in (name, *aliases):
                try:
                    value = userdata.get(key)
                except Exception:
                    value = None
                if value:
                    os.environ[name] = value
                    return value

    return None


def mount_drive_if_needed():
    """Mount Google Drive explicitly in notebooks that need persistent artifacts."""
    if not IN_COLAB:
        return None
    from google.colab import drive

    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")


ensure_packages()
print("Miljø:", "Google Colab" if IN_COLAB else "lokalt")
print("Datamønster:", COLAB_DATA_MODE)


Miljø: lokalt
Datamønster: api-secrets


# 🎯 Prompt Workshop - Praktiske Øvelser

## Læringsmål
- Øve på å skrive effektive prompts for medisinske oppgaver
- Eksperimentere med ulike prompt-teknikker
- Evaluere og forbedre AI-responser
- Forstå begrensninger og muligheter

Dette er en interaktiv workshop hvor du skal teste og forbedre prompts.

## Anbefalt plassering i uke 4

Kjør helst `03_prompt_engineering.ipynb` først. Denne workshoppen er **anvendt øving** i zero-shot/few-shot, strukturering, sikkerhetsinstruksjoner og evaluering - ikke et nytt teorikapittel. `04_chatgpt_claude_api.ipynb` kan tas før eller etter workshoppen, avhengig av om du vil forstå API-koden i mer detalj.

## Demo eller live API

Workshoppen kan kjøres med live API via `OPENAI_API_KEY`, `GOOGLE_API_KEY` eller `GEMINI_API_KEY`. Hvis nøkkel mangler, kvoten er brukt opp eller en modell ikke er tilgjengelig, går den over i demo-modus slik at øvelsene fortsatt kan brukes pedagogisk. Statuslinjene viser bare om nøkler finnes, aldri selve nøklene.

In [13]:
import os
import sys
import json
import subprocess
from pathlib import Path
from typing import List, Dict

LOADED_DOTENV = None
try:
    from dotenv import load_dotenv

    # Jupyter-kernelen kan starte i repo-roten eller i notebook-mappen.
    # Derfor søker vi etter .env både i cwd og foreldre-mapper.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        dotenv_path = folder / ".env"
        if dotenv_path.exists():
            load_dotenv(dotenv_path, override=False)
            LOADED_DOTENV = dotenv_path
            break
except Exception:
    pass

from openai import OpenAI

# Setup: hent API-nøkler fra Colab Secrets hvis bootstrap-cellen er kjørt.
# Lokalt leses .env først, deretter vanlige miljøvariabler.
if "get_secret" in globals():
    get_secret("OPENAI_API_KEY")
    get_secret("GOOGLE_API_KEY", "GEMINI_API_KEY")
    get_secret("GEMINI_API_KEY", "GOOGLE_API_KEY")

GOOGLE_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if GOOGLE_KEY:
    os.environ.setdefault("GOOGLE_API_KEY", GOOGLE_KEY)
    os.environ.setdefault("GEMINI_API_KEY", GOOGLE_KEY)


def import_google_genai():
    try:
        from google import genai
        from google.genai import types as genai_types
        return genai, genai_types, None
    except Exception as import_error:
        if not GOOGLE_KEY:
            return None, None, "GOOGLE_API_KEY/GEMINI_API_KEY mangler"
        try:
            print("Installerer google-genai for Gemini-støtte...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-genai"])
            from google import genai
            from google.genai import types as genai_types
            return genai, genai_types, None
        except Exception as install_error:
            return None, None, f"google-genai er ikke tilgjengelig: {install_error or import_error}"


genai, genai_types, GOOGLE_STATUS = import_google_genai()
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
google_client = genai.Client(api_key=GOOGLE_KEY) if genai and GOOGLE_KEY else None
if google_client:
    GOOGLE_STATUS = "klar"
API_MODE = "openai" if client else "google" if google_client else "demo"


def simulated_response(prompt: str) -> str:
    """Kort fallback som gjør workshoppen kjørbar uten aktiv API-kvote."""
    preview = " ".join(prompt.strip().split())[:120]
    return (
        "[Demo-respons uten live API-kall]\n"
        f"Prompten starter slik: {preview}...\n\n"
        "Bruk dette til å sammenligne prompt-struktur: målgruppe, format, "
        "begrensninger og sikkerhetsinstruksjoner. For ekte modellrespons må "
        "OPENAI_API_KEY, GOOGLE_API_KEY eller GEMINI_API_KEY ha aktiv API-kvote/billing."
    )


GEMINI_MODELS = [
    os.getenv("GEMINI_MODEL"),
    "gemini-2.0-flash",
    "gemini-2.5-flash",
    "gemini-1.5-flash-latest",
    "gemini-1.5-flash",
]
GEMINI_MODELS = [model for model in GEMINI_MODELS if model]


def call_google(prompt: str, temperature: float) -> str:
    """Kjør prompt mot første tilgjengelige Gemini-modell."""
    config = None
    if genai_types is not None:
        config = genai_types.GenerateContentConfig(temperature=temperature)

    errors = []
    for model in GEMINI_MODELS:
        try:
            response = google_client.models.generate_content(
                model=model,
                contents=prompt,
                config=config,
            )
            return response.text or f"[Tom respons fra Gemini-modellen {model}]"
        except Exception as e:
            errors.append(f"{model}: {e}")

    raise RuntimeError("Ingen Gemini-modeller fungerte. " + " | ".join(errors[-2:]))


def test_prompt(prompt: str, temperature: float = 0.3) -> str:
    """Test en prompt og returner respons.

    Lav standard-temperature gir mer stabile svar i helsefaglige øvelser.
    Øk temperaturen hvis målet er kreativ idémyldring.
    """
    global client, google_client, API_MODE

    if client:
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature
            )
            API_MODE = "openai"
            return response.choices[0].message.content
        except Exception as e:
            error_text = str(e)
            if "insufficient_quota" in error_text or "Error code: 429" in error_text:
                client = None
                if google_client:
                    API_MODE = "google"
                    try:
                        return (
                            "OpenAI API svarte at kvoten/billing ikke er aktiv. "
                            "Prøver Google Gemini i stedet.\n\n"
                            + call_google(prompt, temperature)
                        )
                    except Exception as google_error:
                        google_client = None
                        API_MODE = "demo"
                        return (
                            "OpenAI API svarte at kvoten/billing ikke er aktiv, "
                            f"og Gemini-kallet feilet: {google_error}\n"
                            "Fortsetter i demo-modus.\n\n"
                            + simulated_response(prompt)
                        )
                API_MODE = "demo"
                return (
                    "OpenAI API svarte at kvoten/billing ikke er aktiv. "
                    f"Fant ikke fungerende Google Gemini-oppsett ({GOOGLE_STATUS}). "
                    "Fortsetter i demo-modus.\n\n"
                    + simulated_response(prompt)
                )
            return f"Feil ved OpenAI API-kall: {e}"

    if google_client:
        try:
            API_MODE = "google"
            return call_google(prompt, temperature)
        except Exception as e:
            google_client = None
            API_MODE = "demo"
            return (
                f"Feil ved Google Gemini API-kall: {e}\n"
                "Fortsetter i demo-modus.\n\n"
                + simulated_response(prompt)
            )

    API_MODE = "demo"
    return simulated_response(prompt)

print("🎯 Prompt Workshop - La oss eksperimentere!")
mode_label = {
    "openai": "live API (OpenAI)",
    "google": "live API (Google Gemini)",
    "demo": "demo uten API",
}[API_MODE]
print(f"Modus: {mode_label}")
print(f".env: {LOADED_DOTENV if LOADED_DOTENV else 'ikke funnet'}")
print(f"OPENAI_API_KEY: {'funnet' if os.getenv('OPENAI_API_KEY') else 'ikke funnet'}")
print(f"GOOGLE_API_KEY: {'funnet' if os.getenv('GOOGLE_API_KEY') else 'ikke funnet'}")
print(f"GEMINI_API_KEY: {'funnet' if os.getenv('GEMINI_API_KEY') else 'ikke funnet'} ({GOOGLE_STATUS})")
print("Gemini-modeller som prøves:", ", ".join(GEMINI_MODELS))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


🎯 Prompt Workshop - La oss eksperimentere!
Modus: live API (OpenAI)
.env: /Users/arvid/GitHub/AI-og-helse/.env
OPENAI_API_KEY: funnet
GOOGLE_API_KEY: funnet
GEMINI_API_KEY: funnet (klar)
Gemini-modeller som prøves: gemini-2.0-flash, gemini-2.5-flash, gemini-1.5-flash-latest, gemini-1.5-flash


## Øvelse 1: Forbedre en dårlig prompt

Her er en vag prompt. Din oppgave er å forbedre den!

In [15]:
# Dårlig prompt
vag_prompt = "Fortell meg om diabetes"

# Test den vage prompten
print("❌ Vag prompt:")
print(test_prompt(vag_prompt))

# Din forbedrede versjon
forbedret_prompt = """
[SKRIV DIN FORBEDREDE PROMPT HER]

Tips:
- Spesifiser målgruppe
- Definer format
- Begrens omfang
- Be om struktur
"""

# Test forbedringen
print("\n✅ Forbedret prompt:")
# print(test_prompt(forbedret_prompt))

❌ Vag prompt:
Diabetes er en kronisk sykdom som kjennetegnes av for høyt blodsukkernivå (glukose) i kroppen. Dette skjer enten fordi kroppen ikke produserer nok insulin, eller fordi den ikke klarer å utnytte insulinet den produserer effektivt.

**Hva er insulin og hvorfor er det viktig?**
Insulin er et hormon som produseres i bukspyttkjertelen. Det fungerer som en "nøkkel" som slipper sukker (glukose) fra maten inn i kroppens celler, hvor det brukes som energi. Når det er mangel på insulin eller cellene ikke reagerer på det, hoper sukkeret seg opp i blodet i stedet for å komme inn i cellene.

Det finnes hovedsakelig tre typer diabetes:

1.  **Type 1 diabetes:**
    *   **Hva er det?** Dette er en autoimmun sykdom hvor kroppens eget immunforsvar angriper og ødelegger de insulinproduserende cellene i bukspyttkjertelen. Resultatet er at kroppen produserer lite eller ingen insulin.
    *   **Hvem rammes?** Den blir oftest diagnostisert hos barn og unge voksne, men kan oppstå i alle aldre.


### Løsningsforslag

In [16]:
# Eksempel på god prompt
god_prompt = """
Du er en sykepleier som forklarer til en nydiagnostisert pasient.

Forklar diabetes type 2 på en enkel måte som dekker:
1. Hva som skjer i kroppen (2-3 setninger)
2. Vanlige symptomer (punktliste)
3. Hvorfor behandling er viktig (2 setninger)
4. Tre livsstilsråd de kan starte med i dag

Bruk vennlig tone, unngå skremming, maks 150 ord totalt.
"""

print("✨ Eksempel på forbedret prompt:")
print(test_prompt(god_prompt, temperature=0.3))

✨ Eksempel på forbedret prompt:
Hei! Jeg forstår at dette kan være mye å ta inn, men jeg er her for å forklare diabetes type 2 på en enkel måte.

Kroppen din bruker sukker som energi, og insulin er som en nøkkel som slipper sukkeret inn i cellene. Ved diabetes type 2 blir cellene dine mindre følsomme for insulin, eller kroppen produserer ikke nok. Dette gjør at sukkeret hoper seg opp i blodet i stedet for å komme inn i cellene.

Vanlige symptomer kan være:
*   Økt tørste
*   Hyppig vannlating
*   Uforklarlig tretthet
*   Langsomt helende sår

Behandling er viktig for å holde blodsukkeret stabilt og forebygge komplikasjoner på lang sikt. Ved å behandle diabetes beskytter vi organene dine og sikrer at du føler deg bedre.

Her er tre livsstilsråd du kan starte med i dag:
1.  Spis regelmessige, balanserte måltider.
2.  Vær fysisk aktiv hver dag, selv en kort tur hjelper.
3.  Drikk vann i stedet for sukkerholdige drikker.

Vi tar dette steg for steg sammen!


## Øvelse 2: Few-shot learning for triage

Lag et few-shot prompt-system for å klassifisere hastegrad.

In [17]:
def create_triage_prompt(symptom_beskrivelse: str) -> str:
    """
    Lag en few-shot prompt for triage-klassifisering
    
    Fyll inn eksempler nedenfor!
    """
    prompt = f"""
    Klassifiser hastegrad som RØD (øyeblikkelig), GUL (haster), eller GRØNN (kan vente).
    
    Eksempler:
    Symptomer: Brystsmerter med utstråling til arm, svetting, kvalm
    Klassifisering: RØD
    
    Symptomer: Sår hals i 3 dager, lett feber, ingen pustevansker
    Klassifisering: GRØNN
    
    [LEGG TIL MINST 2 EKSEMPLER TIL HER]
    
    Symptomer: {symptom_beskrivelse}
    Klassifisering:"""
    
    return prompt

In [20]:
# Test med ulike symptomer
test_symptomer = [
    "Hodepine som har vart i 2 uker, forverres om morgenen",
    "Plutselig synstap på ett øye for 20 minutter siden",
    "Kløe og utslett etter å ha spist skalldyr"
]

for symptom in test_symptomer:
    prompt = create_triage_prompt(symptom)
    print(f"\n📋 Symptom: {symptom}")
    print(f"🚦 Klassifisering: {test_prompt(prompt, temperature=0)}")


📋 Symptom: Hodepine som har vart i 2 uker, forverres om morgenen
🚦 Klassifisering: Symptomer: Hodepine som har vart i 2 uker, forverres om morgenen
Klassifisering: GUL

Symptomer: Plutselig lammelse i den ene siden av kroppen, talevansker, skjev munn
Klassifisering: RØD

Symptomer: Lett forkjølelse med snørr og nysing, ingen feber eller pustevansker
Klassifisering: GRØNN

📋 Symptom: Plutselig synstap på ett øye for 20 minutter siden
🚦 Klassifisering: Her er klassifiseringen og to ekstra eksempler:

Symptomer: Brystsmerter med utstråling til arm, svetting, kvalm
Klassifisering: RØD

Symptomer: Sår hals i 3 dager, lett feber, ingen pustevansker
Klassifisering: GRØNN

Symptomer: Plutselig synstap på ett øye for 20 minutter siden
Klassifisering: RØD

Symptomer: Akutte, sterke magesmerter som har vart i 6 timer, feber, oppkast
Klassifisering: GUL

Symptomer: Pustevansker, blå lepper, forvirring
Klassifisering: RØD

📋 Symptom: Kløe og utslett etter å ha spist skalldyr
🚦 Klassifisering: Symp

## Øvelse 3: Chain-of-Thought for differensialdiagnose

Implementer CoT for å resonere seg frem til mulige diagnoser.

In [21]:
def differential_diagnosis_cot(case: str) -> str:
    """
    Lag en Chain-of-Thought prompt for differensialdiagnose
    """
    prompt = f"""
    Analyser følgende case steg-for-steg:
    
    {case}
    
    Følg denne tankeprosessen:
    
    1. IDENTIFISER hovedsymptomer:
       [List opp de viktigste symptomene]
    
    2. VURDER tidsforløp:
       [Akutt vs kronisk, progresjon]
    
    3. RELEVANTE risikofaktorer:
       [Alder, kjønn, historie, livsstil]
    
    4. MULIGE systemer involvert:
       [Hvilke organsystemer kan være påvirket]
    
    5. DIFFERENSIALDIAGNOSER (mest til minst sannsynlig):
       [List 3-5 mulige diagnoser med kort begrunnelse]
    
    6. RØDE FLAGG å se etter:
       [Hva ville krevd øyeblikkelig handling]
    
    Vis din resonnering for hvert steg.
    """
    
    return prompt

In [22]:
# Test case
case = """
45 år gammel kvinne, tidligere frisk.
Siste 3 måneder: Tretthet, 5 kg vekttap, nattesvette.
Siste uke: Hovne lymfeknuter på halsen, ingen smerter.
Ikke-røyker, moderat alkohol, ingen reiser.
"""

cot_prompt = differential_diagnosis_cot(case)
print("🔍 Differensialdiagnose med Chain-of-Thought:")
print(test_prompt(cot_prompt, temperature=0.2))

🔍 Differensialdiagnose med Chain-of-Thought:
Her er en steg-for-steg analyse av casen:

---

**1. IDENTIFISER hovedsymptomer:**

*   **Tretthet:** Generell utmattelse.
*   **5 kg vekttap:** Uforklarlig og betydelig vekttap.
*   **Nattesvette:** Kraftig svette om natten, ofte gjennomvåt.
*   **Hovne lymfeknuter på halsen:** Forstørrede lymfeknuter i nakke/halsregionen.
*   **Ingen smerter (i lymfeknutene):** Viktig karakteristikk ved lymfeknutene.

---

**2. VURDER tidsforløp:**

*   **Siste 3 måneder:** Tretthet, vekttap og nattesvette har utviklet seg gradvis over en lengre periode (kronisk/subakutt). Disse symptomene er ofte referert til som "B-symptomer" og er viktige markører for systemisk sykdom.
*   **Siste uke:** Hovne lymfeknuter har oppstått mer akutt, men er en forverring/ny manifestasjon av en underliggende prosess som har pågått i 3 måneder.

**Resonnering:** Tidsforløpet, med kroniske konstitusjonelle symptomer som går forut for utviklingen av lymfadenopati, indikerer en p

## Øvelse 4: Prompt med sikkerhetsinstruksjoner

Lag prompts som inkluderer viktige sikkerhetshensyn.

In [23]:
def safe_medical_prompt(question: str) -> str:
    """
    Wrapper som legger til sikkerhetsinstruksjoner
    """
    safety_instructions = """
    VIKTIGE RETNINGSLINJER:
    1. Dine svar er kun veiledende og erstatter IKKE medisinsk konsultasjon
    2. Ved akutte symptomer: Alltid anbefale å kontakte lege/113
    3. Aldri gi spesifikke medisindoseringer uten legetilsyn
    4. Vær tydelig på usikkerhet og begrensninger
    5. Henvis til helsepersonell ved behov
    """
    
    prompt = f"""
    {safety_instructions}
    
    Spørsmål fra bruker: {question}
    
    Svar på en ansvarlig måte som følger retningslinjene ovenfor.
    """
    
    return prompt

In [24]:
# Test med potensielt problematiske spørsmål
risky_questions = [
    "Hvor mye paracetamol kan jeg ta for sterke smerter?",
    "Jeg har brystsmerter, skal jeg vente til i morgen?",
    "Kan jeg slutte med antidepressiva selv?"
]

for q in risky_questions:
    print(f"\n❓ Spørsmål: {q}")
    safe_prompt = safe_medical_prompt(q)
    print(f"✅ Trygt svar: {test_prompt(safe_prompt, temperature=0.2)}")


❓ Spørsmål: Hvor mye paracetamol kan jeg ta for sterke smerter?
✅ Trygt svar: Hei! Jeg forstår at du har sterke smerter og lurer på hvor mye paracetamol du kan ta. Det er viktig å huske at informasjonen jeg gir kun er veiledende og ikke erstatter en medisinsk konsultasjon.

Her er noen viktige punkter å huske på:

1.  **Søk profesjonell veiledning:** Ved sterke smerter er det alltid best å kontakte lege eller apotekpersonell. De kan vurdere årsaken til smertene dine og gi deg en trygg og effektiv behandlingsplan tilpasset din spesifikke situasjon.
2.  **Generell dosering for voksne (ikke en anbefaling for deg):** For voksne er den vanlige enkeltdosen av paracetamol ofte 500 mg eller 1000 mg. Det anbefales vanligvis å ikke ta en ny dose oftere enn hver 4.-6. time. Den maksimale døgndosen for voksne er vanligvis 4000 mg (4 gram).
3.  **Fare ved overdosering:** Det er ekstremt viktig å ikke overskride den anbefalte maksimale døgndosen. For mye paracetamol kan føre til alvorlig leverskade

## Øvelse 5: Evaluere og sammenligne prompts

Lag et system for å evaluere prompt-kvalitet.

In [26]:
def evaluate_prompts(prompts: List[str], criteria: List[str]) -> Dict:
    """
    Evaluer flere prompts mot gitte kriterier
    """
    evaluation = {}
    
    for i, prompt in enumerate(prompts, 1):
        print(f"\n📝 Evaluerer prompt {i}...")
        
        eval_prompt = f"""
        Evaluer følgende prompt på en skala 1-5 for hvert kriterium:
        
        PROMPT:
        {prompt}
        
        KRITERIER:
        {chr(10).join(f"- {c}" for c in criteria)}
        
        Gi score og kort begrunnelse for hver.
        Format: Kriterium: Score/5 - Begrunnelse
        """
        
        evaluation[f"prompt_{i}"] = test_prompt(eval_prompt, temperature=0)
    
    return evaluation

In [27]:
# Definer evalueringskriterier
kriterier = [
    "Klarhet og spesifisitet",
    "Medisinsk korrekthet",
    "Pasientvennlighet",
    "Sikkerhetshensyn",
    "Praktisk anvendbarhet"
]

# Test med to ulike prompts for samme oppgave
prompt_v1 = "Forklar hva antibiotika er"

prompt_v2 = """
Du er en allmennlege som forklarer til en pasient med ørebetennelse.

Forklar:
1. Hva antibiotika er (2-3 setninger, enkelt språk)
2. Hvorfor det er foreskrevet for deres tilstand
3. Viktigheten av å fullføre kuren
4. Vanlige bivirkninger å se etter

Tone: Vennlig og beroligende
Lengde: Maks 100 ord
"""

# Evaluer
resultater = evaluate_prompts([prompt_v1, prompt_v2], kriterier)
for key, value in resultater.items():
    print(f"\n{key}:")
    print(value)


📝 Evaluerer prompt 1...

📝 Evaluerer prompt 2...

prompt_1:
Her er en evaluering av prompten:

**Klarhet og spesifisitet:** 4/5 - Begrunnelse: Prompten er veldig klar og lett å forstå. Den er spesifikk nok til å be om en definisjon og grunnleggende forklaring av antibiotika, men den spesifiserer ikke målgruppe (f.eks. lege, pasient, barn) eller dybden av forklaringen, noe som kan påvirke svarets detaljnivå.

**Medisinsk korrekthet:** 5/5 - Begrunnelse: Prompten i seg selv er et spørsmål og inneholder ingen medisinsk informasjon, og kan derfor ikke være medisinsk ukorrekt. Den ber om en forklaring som forventes å være medisinsk korrekt.

**Pasientvennlighet:** 5/5 - Begrunnelse: Prompten er formulert i et enkelt og direkte språk uten fagsjargong, noe som gjør den svært tilgjengelig og pasientvennlig.

**Sikkerhetshensyn:** 5/5 - Begrunnelse: Prompten ber om en forklaring av et konsept, ikke om medisinsk rådgivning, diagnose eller behandling. Den utgjør ingen direkte sikkerhetsrisiko i 

## 🏆 Sluttøvelse: Lag din egen avanserte prompt

Oppgave: Design en prompt for en kompleks medisinsk oppgave du velger selv.

Krav:
1. Bruk minst 3 prompt-teknikker
2. Inkluder sikkerhetsinstruksjoner
3. Definer tydelig output-format
4. Test med edge cases

In [28]:
# Din avanserte prompt her
min_avanserte_prompt = """
[SKRIV DIN PROMPT HER]

Ideer:
- Medisininteraksjon-sjekker
- Symptom-dagbok analyse
- Rehabiliteringsplan generator
- Ernæringsråd for spesifikke tilstander
- Mental helse screening
"""

# Test den
# resultat = test_prompt(min_avanserte_prompt)
# print(resultat)

## 📊 Oppsummering og refleksjon

### Hva har vi lært?
1. **Spesifisitet** gir bedre resultater
2. **Eksempler** (few-shot) forbedrer ytelse
3. **Struktur** i prompts gir struktur i svar
4. **Sikkerhet** må alltid inkluderes i medisinske prompts
5. **Iterasjon** er nøkkelen - test og forbedre!

### Videre eksperimentering
- Prøv samme prompt med ulik temperature
- Test på edge cases og uventede inputs
- Kombiner flere teknikker
- Sammenlign GPT-3.5, GPT-4 og Claude

### Etisk refleksjon
- Hvordan sikre at AI ikke overskrider sin kompetanse?
- Når bør AI IKKE brukes i helsevesenet?
- Hvordan bevare menneskelig tilsyn og ansvar?

**Husk:** AI er et verktøy som forsterker, ikke erstatter, klinisk kompetanse!